# Severity SVC — simple hyperparameter search

Uses **train.csv only** and the production feature definition: first derivative (`d1`) → `StandardScaler` → RBF SVC. Validation is grouped by `engine_id`; `test.csv` is never loaded.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'train.csv').exists(): ROOT = ROOT.parent
TRAIN = ROOT / 'data' / 'train.csv'
df = pd.read_csv(TRAIN)
print('train:', df.shape)

In [ ]:
faults = ['zakoksowany', 'lejacy', 'pompa', 'iglica']
severity_map = {'male': 0, 'srednie': 1, 'duze': 2}
work = df[df['label'].isin(faults) & df['severity'].isin(severity_map)].copy()
X_raw = work[[f'mV_{i}' for i in range(21)]].apply(pd.to_numeric, errors='coerce').fillna(0.0).to_numpy()
X = np.diff(X_raw, axis=1)
y = work['severity'].map(severity_map).to_numpy()
groups = work['engine_id'].astype(str).to_numpy()
print('severity rows:', len(work))
print('engines:', len(np.unique(groups)))
print(work['severity'].value_counts())

In [ ]:
cv = GroupKFold(n_splits=min(5, len(np.unique(groups))))
pipe = Pipeline([('scaler', StandardScaler()), ('svc', SVC(kernel='rbf', probability=True))])
param_grid = {
    'svc__C': [0.1, 0.3, 1, 3, 10, 30, 100],
    'svc__gamma': ['scale', 0.001, 0.003, 0.01, 0.0175, 0.03, 0.1],
    'svc__class_weight': [None, 'balanced'],
}
search = GridSearchCV(pipe, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, return_train_score=False, refit=True, verbose=1)
search.fit(X, y, groups=groups)
print('BEST:', search.best_params_)
print('BEST CV F1 macro:', search.best_score_)

In [ ]:
results = pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
display(results[['rank_test_score','mean_test_score','std_test_score','param_svc__C','param_svc__gamma','param_svc__class_weight']].head(20))

In [ ]:
report = {'model':'RBF SVC','features':'first derivative (d1)','cv':'GroupKFold by engine_id','best_params':{k.replace('svc__',''):v for k,v in search.best_params_.items()},'best_f1_macro':float(search.best_score_)}
out = ROOT / 'reports' / 'severity_notebook_result.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
print(out)